In [5]:
from pyspark.sql.functions import *
from pyspark.sql.session import SparkSession

spark = (
    SparkSession.builder.master("local[*]")
    .appName("notebook_teste")
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.3.4,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.261,"
        "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.7.1,"
        "org.scala-lang:scala-library:2.13.12",
    )
    # Configuração do Catálogo Iceberg
    .config("spark.sql.catalog.icebergtb", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.driver.extraJavaOptions", "-Dlog4j.configuration=log4j.properties")
    .config("spark.log.level", "ERROR")
    .config("spark.eventLog.enabled", "false")
    # .config("spark.sql.catalog.icebergtb.catalog-impl", "org.apache.iceberg.hadoop.HadoopCatalog")
    .config("spark.sql.catalog.icebergtb.type", "hadoop")
    .config("spark.sql.catalog.icebergtb.warehouse", "s3a://iceberg/")
    .config("spark.sql.catalog.icebergtb.hadoop.fs.s3a.endpoint", "http://localhost:9000")
    .config("spark.sql.catalog.icebergtb.hadoop.fs.s3a.access.key", "minioadmin")
    .config("spark.sql.catalog.icebergtb.hadoop.fs.s3a.secret.key", "minioadmin")
    .config("spark.sql.catalog.icebergtb.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.sql.catalog.icebergtb.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.executor.extraJavaOptions", "-Dio.netty.tryReflectionSetAccessible=true")
    .config("spark.driver.extraJavaOptions", "-Dio.netty.tryReflectionSetAccessible=true")
    .getOrCreate()
)

spark

In [6]:
currency_rate_df = spark.sql(""" select  
                                    row_number() over(order by currencyrateid) as currencyrate_key,
                                    * 
                             
                                from icebergtb.advworks_rawdb.raw_dimcurrency_rate """)

In [7]:
currency_rate_df.show()

+----------------+--------------+----------------+--------------+-----------+------------+----------------+--------------------+--------------------+----------+
|currencyrate_key|currencyrateid|fromcurrencycode|tocurrencycode|averagerate|endofdayrate|currencyratedate|       inserted_date|       modified_date|eltversion|
+----------------+--------------+----------------+--------------+-----------+------------+----------------+--------------------+--------------------+----------+
|               1|             1|             USD|           ARS|     1.0000|      1.0002|      2011-05-31|2025-03-01 12:05:...|9999-31-12 00:00:...|         1|
|               2|             2|             USD|           AUD|     1.5491|      1.5500|      2011-05-31|2025-03-01 12:05:...|9999-31-12 00:00:...|         1|
|               3|             3|             USD|           BRL|     1.9379|      1.9419|      2011-05-31|2025-03-01 12:05:...|9999-31-12 00:00:...|         1|
|               4|             4| 